# First proper model: multinomial logistic regression

Train sparse multinomial logistic-regression candidates, select using validation macro-F1, and evaluate once on the untouched test partition.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import baselines
baselines = importlib.reload(baselines)
from baselines import MajorityClassClassifier
from model_data import prepare_model_data
from model_evaluation import evaluate_models
from models import make_multinomial_logistic_regression

## Prepare chronological model partitions

`prepare_model_data` fits the feature transformer on training rows only and applies it to validation and test rows.

In [2]:
model_data = prepare_model_data(PROJECT_ROOT / "data")

pd.DataFrame({
    "partition": ["train", "validation", "test"],
    "rows": [
        len(model_data.y_train),
        len(model_data.y_validation),
        len(model_data.y_test),
    ],
    "encoded_features": [
        model_data.X_train.shape[1],
        model_data.X_validation.shape[1],
        model_data.X_test.shape[1],
    ],
})

partition     rows  encoded_features
0       train  9864275              4758
1  validation  2120356              4758
2        test  2126112              4758

## Select regularization with validation macro-F1

Candidates are fitted only on the training partition. The validation partition is used for model selection.

In [ ]:
selected_name = "logistic_regression_C_0.01"
selected_model = make_multinomial_logistic_regression(C=0.01)
selected_model.fit(model_data.X_train, model_data.y_train)

## Regularization search result

The following candidates were fitted on the training partition and compared on the chronological validation partition:

```python
C_values = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
```

| Model | Validation rows | Macro-F1 | Accuracy |
|---|---:|---:|---:|
| logistic_regression_C_0.01 | 2,120,356 | 0.309290 | 0.375253 |
| logistic_regression_C_0.03 | 2,120,356 | 0.308532 | 0.373045 |
| logistic_regression_C_0.1 | 2,120,356 | 0.307921 | 0.371345 |
| logistic_regression_C_0.3 | 2,120,356 | 0.307589 | 0.370564 |
| logistic_regression_C_1.0 | 2,120,356 | 0.307444 | 0.370225 |
| logistic_regression_C_3.0 | 2,120,356 | 0.307429 | 0.370133 |
| logistic_regression_C_10.0 | 2,120,356 | 0.307414 | 0.370100 |

`C=0.01` achieved the highest validation macro-F1, so it is retained as the selected model. The test partition was not used for this choice.

## Compare with the majority baseline on test data

The test partition is used only after model selection.

In [ ]:
majority_model = MajorityClassClassifier().fit(
    model_data.X_train,
    model_data.y_train,
)

test_results = evaluate_models(
    {
        "majority_baseline": majority_model,
        selected_name: selected_model,
    },
    model_data.X_test,
    model_data.y_test,
)
test_results

## First test results

Logistic regression with `C=0.01` achieved a test macro-F1 of **0.3042**, compared with **0.1234** for the majority baseline. Its accuracy was lower (**0.3751** versus **0.4463**) because it predicts minority delay classes instead of mainly choosing the most common class. Macro-F1 is therefore the preferred comparison metric.

## Alternative linear model results

A quick validation comparison used the same chronological training and validation partitions. Logistic regression remained the strongest model by macro-F1, so it is retained as the project linear model.

| Model | Validation macro-F1 | Validation accuracy |
|---|---:|---:|
| Logistic regression (`C=0.01`) | 0.309290 | 0.375253 |
| LinearSVC (`C=0.01`) | 0.305399 | 0.428837 |
| ComplementNB | 0.287530 | 0.405584 |
| SGD hinge | 0.267145 | 0.344444 |

The alternatives did not improve macro-F1, the primary metric for this imbalanced classification task.